# Handling file inputs AND outputs: A notebook on the spaCy similarity readings

* This notebook is sharing code on exploring spaCy's "token similarity" readings in different text inputs based on a word/token of interest.
* This notebook also helps display / explore how to output a variety of output media types from a Python dictionary. 
* And hey, it introduces a code notebook to help document code.


In [1]:
!pip install spacy
import spacy
import os, shutil

# FOR XML OUTPUT:  install dicttoxml
!pip install dicttoxml
from dicttoxml import dicttoxml
# install xml.dom.minidom
from xml.dom.minidom import parseString

# FOR DATAFRAMES OUTPUT
import pandas as pd

# FOR JSON OUTPUT
import json

Above are all the package imports. 
* What is `os`? This is a library that allows Python to read your operating system (os).
This will let Python read and interpret file paths on your local computer.
* `shutil` will let us remove unwanted files from a directory. 
* We'll use the others to help us package up various file outputs.

In [2]:
# LOAD THE SPACY LANGUAGE MODEL: choices are _sm, _md, _lg, but _sm won't work here.
nlp = spacy.cli.download("en_core_web_md")
nlp = spacy.load('en_core_web_md')


AFTER THE FIRST DOWNLOAD, COMMENT OUT the spacy.cli.download(...) variable.
* Your spaCy language model will already be stored in your Python environment.
* ABOUT WHAT SPACY SHOULD LOAD: Try en_core_web_md
    * There are _sm, _md, and _lg models built into spaCy. Each takes up more space than the others, but contains more data, so may be more accurate/precise.
    * If we try the sm model, we're told that it does not have word vectors loaded, so it uses tagger, parser and NER (named entity recognition to calculate similarity instead. 
    * Better to use to the _md or _lg models, and you can compare the results of each.

## OBJECTIVE: Find out which words in each document are most similar to a particular word of interest!

*  First, we need to set up Python to read input files from a directory
* We'll use `os` to identify a file directory with text files to explore:
* `os.cwd` returns the **current working directory path** (where you're saving your Python file)!

In [3]:
workingDir = os.getcwd()
print("current working directory: " + workingDir)

# os.listdir lists files and folders inside a path:
insideDir = os.listdir(workingDir)
print("inside this directory are the following files AND directories: " + str(insideDir))

# use os.path.join to connect the subdirectory to the working directory:
CollPath = os.path.join(workingDir, 'textCollection')
print(f'{CollPath=}')

current working directory: /Users/eeb4/Documents/GitHub/newtfire/textAnalysis-Hub/Class-Examples/Python/readFileCollections-example
inside this directory are the following files AND directories: ['JSON-output', '.DS_Store', 'csv-output', 'rdColl-and-Outputs.ipynb', 'readingFileCollection.py', 'xml-output', 'similarityReadings.txt', 'rdColl-and-Outputs.py', 'textCollection', '.ipynb_checkpoints']
CollPath='/Users/eeb4/Documents/GitHub/newtfire/textAnalysis-Hub/Class-Examples/Python/readFileCollections-example/textCollection'


See the last `print` line? I'm using a "pretty-formatted" print called the "f-string" that lets me quickly identify a variable AND what it returns. `print(f'{CollPath=}')` I'm going to keep using this throughout my code because it helps test the variables as we go! 

## BIG NLP FUNCTION WITH FILE INPUT!
The next cell contains a LOT of code! 
* We have our function for reading the text files and performing spaCy's NLP functions on them. 
* AND we include the file input that *feeds* the function. This has to come after the function, because the function needs to be defined **before** you can call it to run over files.

In [ ]:
def readTextFiles(filepath):
    with open(filepath, 'r', encoding='utf8') as f:
        readFile = f.read()
        # print(readFile)
        stringFile = str(readFile)
        lengthFile = len(readFile)
        print(lengthFile)
        # ebb: add that utf8 encoding argument to the open() function to ensure that reading works on everyone's systems
        # this all succeeds if you see the text of your files printed in the console.
        tokens = nlp(stringFile)
        # playing with vectors here
        vectors = tokens.vector

        wordOfInterest = nlp(u'panic')
        # print(wordOfInterest, ': ', wordOfInterest.vector_norm)

        # Now, let's open an empty dictionary! We'll fill it up with the for loop just after it.
        # The for-loop goes over each token and gets its values
        highSimilarityDict = {}
        for token in tokens:
            if (token and token.vector_norm):
                # if token not in highSimilarityDict.keys(): # Alas, this did not work to remove duplicates from my dictionary. :-(
                if wordOfInterest.similarity(token) > .3:
                    highSimilarityDict[token] = wordOfInterest.similarity(token)
                    # The line above creates the structure for each entry in my dictionary.
                    # print(token.text, "about this much similar to", wordOfInterest, ": ", wordOfInterest.similarity(token))
        print("This is a dictionary of words most similar to the word " + wordOfInterest.text + " in this file.")
        print(highSimilarityDict)

        # *********  REMOVE DUPLICATES FROM THE DICTIONARY *************
        # When we print the high similarity dictionary, there are almost certainly duplicate entries.
        #  In this notebook, we'll use the DICTIONARY COMPREHENSION to remove those duplicates!
        #   * Our method pulls a switcheroo! Keys become values in the first dictionary comprehension."
        #   * 'Deduping' happens because when a duplicate token is read as a *value* it is removed.Then we reverse again and make the tokens be keys in the shorter dictionary."
        #   * Source: https://tutorial.eyehunts.com/python/python-remove-duplicates-from-dictionary-example-code/ "
        # *********************************************************************
        switcheroo = {val: key for key, val in highSimilarityDict.items()}
        deduped = {val: key for key, val in switcheroo.items()}
        print(str(len(switcheroo)) + ' **** ' + f'{switcheroo=}')

        print(len(deduped), ' **** ', f'{deduped=}')
        print(len(deduped.items()), " vs ", len(highSimilarityDict.items()))
        
        # ********* SORTING THE VALUES ************ 
        # The sorted() function pull the dictionary items out and sorts by value. 
        # We set reverse=True so that highest similarity values come out first.

        sortedSimValues = sorted(deduped.items(), key=lambda x: x[1], reverse=True)
        print(type(sortedSimValues), f'{sortedSimValues=}')
        # HEY, that's not a dictionary! It's a list. Let's convert it back to a dictionary.

        sortedSimDict = dict(sortedSimValues)
        print(type(sortedSimValues), f'{sortedSimValues=}')
        
        return sortedSimDict
        # The function is done now! It RETURNS the sortedSimValues dictionary based on the files fed to it. 

        # ebb: This controls our file handling as a for loop over the directory:
with open('similarityReadings.txt', 'a', encoding='utf8') as f:
    for file in sorted(os.listdir(CollPath)):
        # My filenames are numbered, so I controlled the order of the for loop by sorting them.
        if file.endswith(".txt"):
            filepath = f"{CollPath}/{file}"
            print(filepath)
            similarityData = readTextFiles(filepath)
            
            # LET'S WRITE OUR DICTIONARY TO A NICE TEXT OUTPUT FILE WHILE WE'RE WORKING ON FILE INPUTS / OUTPUTS!
            f.write(filepath + '\n')
            f.write(str(similarityData) + '\n\n')

   ## FILE OUTPUTS FOR DICTIONARY DATA
        In the previous cell, we set Python to read file inputs, AND we output one text file: similarityReadings.txt
        There are several ways we can output the data in a Python dictionary that store the actual data as packaged data, rather than just a string. The next cell is designed to show different output strategies. 
        

In [ ]:
# EMPTY and CREATE THREE OUTPUT FOLDERS:
# First check and see, do these folders exist? If so, remove them
# because Python won't allow you to overwrite them
if os.path.exists("JSON-output"):
    shutil.rmtree("JSON-output")
if os.path.exists("csv-output"):
    shutil.rmtree("csv-output")
if os.path.exists("xml-output"):
    shutil.rmtree("xml-output")
os.mkdir('JSON-output')
os.mkdir('csv-output')
os.mkdir('xml-output')

for file in sorted(os.listdir(CollPath)):
    # My filenames are numbered, so I controlled the order of the for loop by sorting them.
    if file.endswith(".txt"):
        filepath = f"{CollPath}/{file}"
        print(filepath)
        filenameTxt = os.path.basename(filepath).split('/')[-1]
        filename = filenameTxt[:-4]
        print(filename)
        similarityData = readTextFiles(filepath)

        # ============================================ #
        # JSON: DICTIONARY OUTPUT METHOD 2
        # ============================================ #
        # JSON stands for JavaScript Object Notation
        # It's an adaptable file serialization format for dictionary structures used for web programming
        stringKeys = {str(key): val for key, val in similarityData.items()}
        print(f'{stringKeys=}')
        with open(f'JSON-output/{filename}.json', 'w') as fp:
            JSON = json.dumps(stringKeys)
            print(f'{JSON=}')
            json.dump(stringKeys, fp)

        # ============================================ #
        # PANDAS DATA FRAMES: DICTIONARY OUTPUT METHOD 3
        # ============================================ #
        # Data Frames are used heavily in data analytics.
        # They make a tabular (table) structure and are easily output as a CSV or TSV text file
        df = pd.DataFrame.from_dict(similarityData.items(), orient="columns")
        df.columns = ['token', 'similarity']
        print(df)
        df.to_csv(f'csv-output/{filename}.tsv', sep='\t', index=False, encoding='utf-8')
        # I want to make a tsv file (tab-separated values), so I'm using the \t here.
        # to make a comma-separated values csv, put in a ','

    # ============================================ #
    # XML: DICTIONARY OUTPUT METHOD 4
    # ============================================ #
    # This tutorial is good: https://www.geeksforgeeks.org/serialize-python-dictionary-to-xml/
        xml = dicttoxml(similarityData)
        dom = parseString(xml)
        # dom is just a string. We can pretty print it in the console,
        # but this is not good for writing to an XML file.
        print(dom.toprettyxml())
        with open(f'xml-output/{filename}.xml', 'w') as xmlFile:
            xml_decode = xml.decode()
            xmlFile.write(xml_decode)
            xmlFile.close()
